# Triton Kernel 主线 · 第 9/10 课：Blocked Matmul 与 FP32 累加

> 状态：**参考答案版**  
> 本仓库采用逐课通过制。本课未通过前，不应直接进入下一课。

## 本课目标与完成标准

学完后你应能：实现带 K 尾块的 blocked matmul，解释 accumulator、mask 和 autotune 维度。

通过必须同时满足：

- 独立补齐本课唯一的代码填空题，并通过给定检查；
- 三个问答题均说明因果链，而不是只报术语；
- 能指出至少一个正确性边界和一个性能取舍；
- 总分不低于 8/10，且没有一票否决级概念错误。

## 前置关系

- 课程前置：Python、PyTorch 张量、CUDA 基本线程/内存概念
- 本课在路线中的作用：Triton GEMM 用二维 program grid 负责 C tile，在 K 维循环加载 A/B tiles 并用 `tl.dot` 累加。

## 核心心智模型

### 1. 它是什么，解决什么问题

Triton GEMM 用二维 program grid 负责 C tile，在 K 维循环加载 A/B tiles 并用 `tl.dot` 累加。

### 2. 它如何工作

A mask 覆盖 M/K，B mask 覆盖 K/N；累加器用 FP32，最终按 C dtype 写回。

### 3. 正确性条件与常见误区

每个边界维都要独立 mask；容差必须符合输入/累加精度，不能与 CPU 双精度逐位比较。

### 4. 性能与工程取舍

BM/BN/BK、warps、stages 共同影响 tensor-core 利用率与资源；需要按 shape autotune。

## 具体演示

M=17,K=33,N=19、tile=32 时 M/N/K 都有尾块，是高价值正确性用例。

请在阅读后先合上这一节，用自己的语言复述“输入状态 → 中间状态 → 输出状态”，再做练习。

## 实践任务：唯一代码填空题

补齐 K-tile 累加。

规则：只能修改 `TODO`/`______` 所在位置；不要删除断言或放宽误差。代码注释说明了每个边界条件。

In [ ]:
import torch
import triton
import triton.language as tl

@triton.jit
def matmul_kernel(a, b, c, M: tl.constexpr, N: tl.constexpr, K: tl.constexpr,
                  sam: tl.constexpr, sak: tl.constexpr,
                  sbk: tl.constexpr, sbn: tl.constexpr,
                  scm: tl.constexpr, scn: tl.constexpr,
                  BM: tl.constexpr, BN: tl.constexpr, BK: tl.constexpr):
    rm = tl.program_id(0) * BM + tl.arange(0, BM)
    rn = tl.program_id(1) * BN + tl.arange(0, BN)
    acc = tl.zeros((BM, BN), tl.float32)
    for k0 in range(0, K, BK):
        rk = k0 + tl.arange(0, BK)
        av = tl.load(a + rm[:, None] * sam + rk[None, :] * sak,
                     mask=(rm[:, None] < M) & (rk[None, :] < K), other=0.0)
        bv = tl.load(b + rk[:, None] * sbk + rn[None, :] * sbn,
                     mask=(rk[:, None] < K) & (rn[None, :] < N), other=0.0)
        acc += ______  # TODO: tile 矩阵乘
    tl.store(c + rm[:, None] * scm + rn[None, :] * scn, acc,
             mask=(rm[:, None] < M) & (rn[None, :] < N))

def matmul(a, b):
    assert a.ndim == b.ndim == 2 and a.shape[1] == b.shape[0]
    M, K = a.shape; N = b.shape[1]
    c = torch.empty((M, N), device=a.device, dtype=a.dtype)
    grid = (triton.cdiv(M, 32), triton.cdiv(N, 32))
    matmul_kernel[grid](a, b, c, M, N, K, *a.stride(), *b.stride(), *c.stride(),
                        BM=32, BN=32, BK=32)
    return c

for shape in ((17, 33, 19), (64, 64, 64)):
    M, K, N = shape
    a = torch.randn((M,K), device="cuda", dtype=torch.float16)
    b = torch.randn((K,N), device="cuda", dtype=torch.float16)
    torch.testing.assert_close(matmul(a,b), a@b, atol=2e-2, rtol=2e-2)


### 检查方法

在 CUDA/Triton 环境运行本单元格；断言覆盖规则尺寸和非规则尾块。首次 JIT 不计入性能。

提交时请给出：补齐后的代码、实际运行输出（环境不可用时注明“仅静态审查”）以及对失败用例的解释。

### Q1

不要背定义：请从输入、状态变化和输出三个阶段解释“Blocked Matmul 与 FP32 累加”的工作机制。

**你的答案：**


### Q2

只 mask 输出而不 mask K 尾块会怎样？

**你的答案：**


### Q3

为什么训练 GEMM 与 decode 的 M=1 GEMM 不应共用同一配置？

**你的答案：**


## 评分与通过规则

- 代码 4 分：正常输入 2 分，边界输入 1 分，解释实现 1 分；
- Q1～Q3 各 2 分；
- 一票否决：结果碰巧正确但核心因果链错误、删除边界检查、把未运行结果说成实测。

需要提示时按四级机制请求：概念区域 → 具体方向 → 关键局部 → 完整答案。

## 参考答案（仅 answer 分支）

先完成题目再核对。即使代码一致，也要能解释关键步骤，并尝试更换一个输入规模。

In [ ]:
import torch
import triton
import triton.language as tl

@triton.jit
def matmul_kernel(a, b, c, M: tl.constexpr, N: tl.constexpr, K: tl.constexpr,
                  sam: tl.constexpr, sak: tl.constexpr,
                  sbk: tl.constexpr, sbn: tl.constexpr,
                  scm: tl.constexpr, scn: tl.constexpr,
                  BM: tl.constexpr, BN: tl.constexpr, BK: tl.constexpr):
    rm = tl.program_id(0) * BM + tl.arange(0, BM)
    rn = tl.program_id(1) * BN + tl.arange(0, BN)
    acc = tl.zeros((BM, BN), tl.float32)
    for k0 in range(0, K, BK):
        rk = k0 + tl.arange(0, BK)
        av = tl.load(a + rm[:, None] * sam + rk[None, :] * sak,
                     mask=(rm[:, None] < M) & (rk[None, :] < K), other=0.0)
        bv = tl.load(b + rk[:, None] * sbk + rn[None, :] * sbn,
                     mask=(rk[:, None] < K) & (rn[None, :] < N), other=0.0)
        acc += tl.dot(av, bv)
    tl.store(c + rm[:, None] * scm + rn[None, :] * scn, acc,
             mask=(rm[:, None] < M) & (rn[None, :] < N))

def matmul(a, b):
    assert a.ndim == b.ndim == 2 and a.shape[1] == b.shape[0]
    M, K = a.shape; N = b.shape[1]
    c = torch.empty((M, N), device=a.device, dtype=a.dtype)
    grid = (triton.cdiv(M, 32), triton.cdiv(N, 32))
    matmul_kernel[grid](a, b, c, M, N, K, *a.stride(), *b.stride(), *c.stride(),
                        BM=32, BN=32, BK=32)
    return c

for shape in ((17, 33, 19), (64, 64, 64)):
    M, K, N = shape
    a = torch.randn((M,K), device="cuda", dtype=torch.float16)
    b = torch.randn((K,N), device="cuda", dtype=torch.float16)
    torch.testing.assert_close(matmul(a,b), a@b, atol=2e-2, rtol=2e-2)


### Q1 参考答案

A mask 覆盖 M/K，B mask 覆盖 K/N；累加器用 FP32，最终按 C dtype 写回。

### Q2 参考答案

判断时先检查本课不变量：每个边界维都要独立 mask；容差必须符合输入/累加精度，不能与 CPU 双精度逐位比较。  若不成立，最终数值或系统状态即使暂时正常也不可信。

### Q3 参考答案

迁移时先保证正确性，再比较代价。这里的核心取舍是：BM/BN/BK、warps、stages 共同影响 tensor-core 利用率与资源；需要按 shape autotune。

## 参考资料

- [Triton Tutorials](https://triton-lang.org/main/getting-started/tutorials/)
- [Triton language API](https://triton-lang.org/main/python-api/triton.language.html)

资料用于建立事实基线；面试回答仍需用自己的语言组织。